# Spatial graph analysis across encoders

STAGATE, GraphST, and SpaGCN each build their own spatial graph before learning an embedding. This
notebook extracts and compares those graphs directly, independent of the learned features, to answer a
simple question: *do the three methods actually "see" different spatial neighbourhoods, or do their
clustering differences come from the model rather than the graph?*

**What it does, per DLPFC section:**

1. **Extract each method's spatial neighbourhood** (STAGATE k-NN, GraphST k-NN, SpaGCN distance kernel).
2. **Compute graph statistics** (degree, density, physical edge lengths) for each.
3. **Measure neighbourhood overlap** with the Jaccard index between the discrete graphs (STAGATE vs GraphST).

**Important caveat:** STAGATE and GraphST produce *discrete* neighbour sets, so Jaccard overlap is
directly meaningful. SpaGCN produces a *dense, continuous* Gaussian-weighted affinity matrix, so it is
summarised by its weight distribution instead and is not treated as a k-NN graph. Results are written to
`graph_results/`.

**Takeaway:** once neighbourhood size is controlled, the three graphs are highly similar, which points to
the *encoder* (not the graph) as the source of downstream clustering differences.


## Setup and configuration

Import the three encoder packages and fix the graph hyperparameters: STAGATE and GraphST neighbour counts
(`STAGATE_K`, `GRAPHST_K`) and the SpaGCN percentile (`SPAGCN_P`) used to pick its distance scale.


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import STAGATE
import GraphST
import SpaGCN
import os


sections = [
    "151507", "151508", "151509", "151510",
    "151669", "151670", "151671", "151672",
    "151673", "151674", "151675", "151676"
]

STAGATE_K = 6
GRAPHST_K = 3
SPAGCN_P = 0.5

os.makedirs("graph_results", exist_ok=True)

### Load Sections

In [2]:
def load_section(section):

    adata = sc.read_h5ad(
        f"data/DLPFC_{section}_ST_final.h5ad"
    )

    coords = np.asarray(
        adata.obsm["spatial"]
    )

    return adata, coords

## Stagate extraction


In [3]:
def extract_stagate_neighbors(
    adata,
    k=STAGATE_K
):

    adata_tmp = adata.copy()

    STAGATE.Cal_Spatial_Net(
        adata_tmp,
        k_cutoff=k,
        model="KNN",
        verbose=False
    )

    spatial_net = adata_tmp.uns["Spatial_Net"]

    cell_to_idx = {
        cell: i
        for i, cell in enumerate(
            adata_tmp.obs_names
        )
    }
    neighbors = {
        i: set()
        for i in range(
            adata_tmp.n_obs
        )
    }

    for _, row in spatial_net.iterrows():

        i = cell_to_idx[row["Cell1"]]
        j = cell_to_idx[row["Cell2"]]

        neighbors[i].add(j)

    return neighbors, spatial_net

## GraphST extraction

In [4]:
def extract_graphst_neighbors(
    adata,
    k=GRAPHST_K
):

    adata_tmp = adata.copy()

    GraphST.construct_interaction(
        adata_tmp,
        n_neighbors=k
    )

    adj = np.asarray(
        adata_tmp.obsm["adj"]
    )

    neighbors = {
        i: set(
            np.where(adj[i] > 0)[0]
        )
        for i in range(
            adata_tmp.n_obs
        )
    }

    return neighbors, adj

## SpaGCN distance matrix and weights

In [5]:
def extract_spagcn_distance(
    coords
):

    x = coords[:, 0].tolist()
    y = coords[:, 1].tolist()

    distance_matrix = \
        SpaGCN.calculate_adj_matrix(
            x=x,
            y=y,
            histology=False
        )

    return distance_matrix

def calculate_spagcn_weights(
    distance_matrix,
    p=0.5
):

    # Find SpaGCN's recommended spatial scale
    l = SpaGCN.search_l(
        p,
        distance_matrix
    )

    if l is None:
        raise ValueError(
            "SpaGCN could not find l."
        )

    weights = np.exp(
        -(
            distance_matrix ** 2
        ) / (
            2 * l ** 2
        )
    )

    # Remove self-connections
    np.fill_diagonal(
        weights,
        0
    )

    return weights, l

# def extract_top_k_neighbors(
#     weights,
#     k=3
# ):

#     n = weights.shape[0]

#     neighbors = {}

#     for i in range(n):

#         row = weights[i].copy()

#         # Remove self
#         row[i] = -np.inf

#         # Find k strongest relationships
#         top_k = np.argpartition(
#             row,
#             -k
#         )[-k:]

#         # Sort strongest → weakest
#         top_k = top_k[
#             np.argsort(
#                 row[top_k]
#             )[::-1]
#         ]

#         neighbors[i] = set(
#             top_k.tolist()
#         )

#     return neighbors

## Graph statistics

In [6]:
def calculate_graph_statistics(
    neighbors,
    coords
):

    n_nodes = len(neighbors)

    degrees = np.array([
        len(neighbors[i])
        for i in range(n_nodes)
    ])

    # Unique undirected edges
    edges = set()

    for i, nbrs in neighbors.items():

        for j in nbrs:

            edges.add(
                tuple(
                    sorted((i, j))
                )
            )

    n_edges = len(edges)
    max_edges = (
        n_nodes *
        (n_nodes - 1) /
        2
    )

    density = (
        n_edges /
        max_edges
    )

    # Physical distances
    physical_distances = []

    for i, j in edges:

        distance = np.linalg.norm(
            coords[i] -
            coords[j]
        )

        physical_distances.append(
            distance
        )

    physical_distances = np.asarray(
        physical_distances
    )
    return {
        "nodes": n_nodes,
        "edges": n_edges,
        "average_degree": degrees.mean(),
        "min_degree": degrees.min(),
        "max_degree": degrees.max(),
        "density": density,
        "mean_edge_distance":
            physical_distances.mean(),
        "median_edge_distance":
            np.median(
                physical_distances
            ),
        "min_edge_distance":
            physical_distances.min(),
        "max_edge_distance":
            physical_distances.max(),
        "q25_edge_distance":
            np.percentile(
                physical_distances,
                25
            ),
        "q75_edge_distance":
            np.percentile(
                physical_distances,
                75
            )
    }


## Jaccard neighbourhood overlap

In [7]:
def neighborhood_jaccard(
    neighbors_a,
    neighbors_b,
    n_nodes
):

    scores = []

    for i in range(n_nodes):

        A = neighbors_a[i]
        B = neighbors_b[i]

        union = A | B

        if len(union) == 0:
            continue

        intersection = A & B

        score = (
            len(intersection) /
            len(union)
        )

        scores.append(score)

    return np.asarray(scores)

In [9]:
def calculate_spagcn_statistics(weights, coords):

    n = weights.shape[0]

    # Non-zero weighted relationships
    mask = weights > 0

    # Exclude diagonal
    np.fill_diagonal(mask, False)

    # Number of non-zero directed relationships
    n_connections = np.sum(mask)

    # Weighted degree
    weighted_degree = weights.sum(axis=1)

    # Pairwise weights excluding diagonal
    weight_values = weights[mask]

    # Spatial distances for non-zero relationships
    distance_matrix = np.linalg.norm(
        coords[:, None, :] - coords[None, :, :],
        axis=2
    )

    distances = distance_matrix[mask]

    return {
        "n_spots": n,
        "n_connections": int(n_connections),

        "mean_weight": float(weight_values.mean()),
        "median_weight": float(np.median(weight_values)),
        "min_weight": float(weight_values.min()),
        "max_weight": float(weight_values.max()),

        "mean_weighted_degree":
            float(weighted_degree.mean()),

        "mean_distance":
            float(distances.mean()),
        "median_distance":
            float(np.median(distances)),
    }

## Main experiment


In [12]:
graph_results = []
overlap_results = []
for section in sections:

    print("\n" + "=" * 70)
    print(f"PROCESSING SECTION {section}")
    print("=" * 70)

    # --------------------------------------------------------
    # Load data
    # --------------------------------------------------------

    adata, coords = load_section(
        section
    )

    n_spots = adata.n_obs

    print(
        f"Number of spots: {n_spots}"
    )


    # ========================================================
    # STAGATE
    # ========================================================

    print("\nSTAGATE...")
    stagate_neighbors, stagate_net = \
        extract_stagate_neighbors(
            adata,
        )

    stagate_stats = \
        calculate_graph_statistics(
            stagate_neighbors,
            coords
        )

    graph_results.append({
        "section": section,
        "method": "STAGATE",
        "k": STAGATE_K,
        "l": np.nan,
        **stagate_stats
    })

    # ========================================================
    # GRAPHST
    # ========================================================

    print("GraphST...")

    graphst_neighbors, graphst_adj = \
        extract_graphst_neighbors(
            adata
        )

    graphst_stats = \
        calculate_graph_statistics(
            graphst_neighbors,
            coords
        )

    graph_results.append({
        "section": section,
        "method": "GraphST",
        "k": GRAPHST_K,
        "l": np.nan,
        **graphst_stats
    })

    # ========================================================
    # SPAGCN
    # ========================================================

    print("SpaGCN distance matrix...")

    spagcn_distance = \
        extract_spagcn_distance(
            coords
        )

    print("Searching for SpaGCN l...")

    spagcn_weights, spagcn_l = \
        calculate_spagcn_weights(
            spagcn_distance,
            p=SPAGCN_P
        )

    print(
        f"SpaGCN l = {spagcn_l}"
    )

    # # Top-K representation for controlled
    # # neighborhood comparison
    # spagcn_neighbors = \
    #     extract_top_k_neighbors(
    #         spagcn_weights,
    #         k=K
    #     )

    spagcn_stats = calculate_spagcn_statistics(
        spagcn_weights,
        coords
    )

    graph_results.append({
        "section": section,
        "method": "SpaGCN",
        "k": None,
        "l": spagcn_l,
        **spagcn_stats
    })

    print("Calculating neighborhood overlap...")


    # STAGATE vs GraphST
    j_sg = neighborhood_jaccard(
        stagate_neighbors,
        graphst_neighbors,
        n_spots
    )

    overlap_results.append({
        "section": section,
        "comparison": "STAGATE vs GraphST",
        "mean_jaccard": j_sg.mean(),
        "median_jaccard": np.median(j_sg),
        "std_jaccard": j_sg.std()
    })


    # STAGATE vs SpaGCN
    # j_ss = neighborhood_jaccard(
    #     stagate_neighbors,
    #     spagcn_weights,
    #     n_spots
    # )

    # overlap_results.append({
    #     "section": section,
    #     "comparison": "STAGATE vs SpaGCN",
    #     "mean_jaccard": j_ss.mean(),
    #     "median_jaccard": np.median(j_ss),
    #     "std_jaccard": j_ss.std()
    # })


    # GraphST vs SpaGCN
    # j_gs = neighborhood_jaccard(
    #     graphst_neighbors,
    #     spagcn_weights,
    #     n_spots
    # )

    # overlap_results.append({
    #     "section": section,
    #     "comparison": "GraphST vs SpaGCN",
    #     "mean_jaccard": j_gs.mean(),
    #     "median_jaccard": np.median(j_gs),
    #     "std_jaccard": j_gs.std()
    # })

    np.save(
        f"graph_results/"
        f"spagcn_weights_{section}.npy",
        spagcn_weights
    )


PROCESSING SECTION 151507
Number of spots: 4221

STAGATE...
GraphST...
SpaGCN distance matrix...
Calculateing adj matrix using xy only...
Searching for SpaGCN l...
Run 1: l [0.01, 1000], p [0.0, 312.04922]
Run 2: l [0.01, 500.005], p [0.0, 85.54007]
Run 3: l [0.01, 250.0075], p [0.0, 21.697504]
Run 4: l [0.01, 125.00874999999999], p [0.0, 4.812828]
Run 5: l [0.01, 62.509375], p [0.0, 0.517257]
Run 6: l [31.2596875, 62.509375], p [0.00034475327, 0.517257]
Run 7: l [46.884531249999995, 62.509375], p [0.07717371, 0.517257]
Run 8: l [54.69695312499999, 62.509375], p [0.2438227, 0.517257]
Run 9: l [58.6031640625, 62.509375], p [0.36827576, 0.517257]
Run 10: l [60.556269531249995, 62.509375], p [0.43989897, 0.517257]
Run 11: l [61.532822265625, 62.509375], p [0.47788703, 0.517257]
recommended l =  62.021098632812496
SpaGCN l = 62.021098632812496
Calculating neighborhood overlap...

PROCESSING SECTION 151508
Number of spots: 4380

STAGATE...
GraphST...
SpaGCN distance matrix...
Calculateing 

STAGATE → discrete neighbors
GraphST → discrete neighbors
SpaGCN  → continuous weighted affinities
So Jaccard is directly meaningful for STAGATE vs GraphST, but not directly meaningful between a neighbor set and SpaGCN's entire weighted matrix.

For SpaGCN, we should instead compare its weight distributions / spatial affinity structure.

In [13]:
graph_df = pd.DataFrame(
    graph_results
)

overlap_df = pd.DataFrame(
    overlap_results
)


# ============================================================
# SAVE RESULTS
# ============================================================

graph_df.to_csv(
    "graph_results/"
    "graph_statistics_all_sections.csv",
    index=False
)

overlap_df.to_csv(
    "graph_results/"
    "neighborhood_overlap_all_sections.csv",
    index=False
)

print("\n")
print("=" * 70)
print("GRAPH STATISTICS")
print("=" * 70)

print(
    graph_df.to_string(
        index=False
    )
)


# print("\n")
# print("=" * 70)
# print("NEIGHBORHOOD OVERLAP")
# print("=" * 70)

# print(
#     overlap_df.to_string(
#         index=False
#     )
# )

# summary = (
#     graph_df
#     .groupby("method")
#     [
#         [
#             "edges",
#             "average_degree",
#             "density",
#             "mean_edge_distance",
#             "median_edge_distance"
#         ]
#     ]
#     .agg(["mean", "std"])
# )

# print("\n")
# print("=" * 70)
# print("SUMMARY ACROSS 12 DLPFC SECTIONS")
# print("=" * 70)

# print(summary)
binary_summary = (
    graph_df[
        graph_df["method"].isin(["STAGATE", "GraphST"])
    ]
    .groupby("method")
    [
        [
            "edges",
            "average_degree",
            "density",
            "mean_edge_distance",
            "median_edge_distance"
        ]
    ]
    .agg(["mean", "std"])
)

print("\n")
print("=" * 70)
print("BINARY GRAPH SUMMARY")
print("=" * 70)

print(binary_summary)
spagcn_summary = (
    graph_df[
        graph_df["method"] == "SpaGCN"
    ]
    .groupby("method")
    [
        [
            "mean_weight",
            "median_weight",
            "min_weight",
            "max_weight",
            "mean_weighted_degree",
            "mean_distance",
            "median_distance"
        ]
    ]
    .agg(["mean", "std"])
)

print("\n")
print("=" * 70)
print("SPAGCN WEIGHTED AFFINITY SUMMARY")
print("=" * 70)

print(spagcn_summary)



GRAPH STATISTICS
section  method   k         l  nodes   edges  average_degree  min_degree  max_degree  density  mean_edge_distance  median_edge_distance  min_edge_distance  max_edge_distance  q25_edge_distance  q75_edge_distance  n_spots  n_connections  mean_weight  median_weight   min_weight  max_weight  mean_weighted_degree  mean_distance  median_distance
 151507 STAGATE 6.0       NaN 4221.0 12895.0        6.000000         6.0         6.0 0.001448          142.098411            138.000000         137.000000         364.906837         137.927517         138.423264      NaN            NaN          NaN            NaN          NaN         NaN                   NaN            NaN              NaN
 151507 GraphST 3.0       NaN 4221.0  7248.0        3.434257         3.0         6.0 0.000814          137.851055            137.927517         137.000000         240.002083         137.557261         138.000000      NaN            NaN          NaN            NaN          NaN         NaN       

The graph statistics for SpaGCN in this table are based on its top-3 neighbors, because its native representation is a dense Gaussian-weighted affinity matrix. So don't interpret its edges, degree, or density as the native SpaGCN graph statistics.

I would eventually have two separate tables in the thesis:

Native representation statistics — including SpaGCN's weight distribution.
Controlled top-k neighborhood comparison — where all three methods are reduced to the same k.

That keeps the experiment rigorous and avoids pretending SpaGCN is a kNN method when it isn't.

Investigate 151509
nstead, focus on:

1. Neighborhood agreement

This is where you see meaningful differences.

2. Graph construction mechanism

Why do the methods differ despite starting from essentially the same spatial neighborhood?

3. The outlier sections

Especially 151509 and, to a lesser degree, 151510.

4. Downstream embeddings

This is probably the most important next step.

Although STAGATE, GraphST, and SpaGCN employ different mechanisms for incorporating spatial relationships, their local spatial neighborhoods on the DLPFC dataset are highly similar when the neighborhood size is controlled.

Then the obvious next question becomes:

If they are using almost the same spatial neighborhoods, why do their downstream embeddings and clustering results differ?